## Lista 2
Karol Garbaczewski, Adam Freń

In [162]:
import numpy as np
import scipy.linalg
from operator import itemgetter


### Zadanie 1
Rozwiąż (w pojedynczej precyzji) układ równań

$$A_{5×5}\overrightarrow{x} = \overrightarrow{b}$$
 
gdzie $A_{5×5}$ to macierz Hilberta i $\overrightarrow{b} = (5, 4, 3, 2, 1)^T$ . Skorzystaj z metody
iteracyjnego poprawiania rozwiązań.
#### Rozwiązanie z solverem numpy

In [169]:
n = 5
A = np.array([[1.0 / (i + j - 1) for j in range(1, n + 1)] for i in range(1, n + 1)], dtype=np.float32)
b = np.array([5.0, 4.0, 3.0, 2.0, 1.0], dtype=np.float32).transpose()

x = np.linalg.solve(A, b)
print("Rozwiązanie początkowe x^(1):\n", x)

k=0
while True:
    r = b.astype(np.float64) - np.linalg.matmul(A, x,).astype(np.float64)
    
    res_norm = np.max(np.abs(r))
    print(f"norma reszty = {res_norm}")
    
    if res_norm < 1e-7:  
        break
        
    dx = np.linalg.solve(A, r)
    
    x = x + dx
    k+=1

print("Ostateczne poprawione rozwiązanie:\n", x)

Rozwiązanie początkowe x^(1):
 [   -94.84782   2157.1013  -10697.322    17900.658    -9440.465  ]
norma reszty = 0.0002646446228027344
norma reszty = 6.284576332937775e-05
norma reszty = 6.954437026251981e-13
Ostateczne poprawione rozwiązanie:
 [   -94.84781941   2157.10127276 -10697.32231839  17900.65789665
  -9440.465253  ]


#### Rozwiązanie z naszym solverem

In [164]:
# chcemy zliczac zera pojawiające się na początku macierzy 
def rows_permutations(A):
    """permutuje wiersze tak żeby eliminacja gaussa działała gdy są zera na 1. miejscach. 
        zwraca: 
                new_order - kolejność wierszy po jej zamianie (lista), 
                old_order - permutację odwrotną(lista) """
    zero_ocurrences = []
    rows, columns = A.shape 
    for i in range(rows):
        zero_count = 0 
        for j in range(columns):

            if np.isclose(A[i,j], 0, atol=1e-10):
                zero_count +=1  
            else:
                break 
        zero_ocurrences.append([zero_count, i])

    zero_ocurrences = sorted(zero_ocurrences, key= lambda x: x[0])
    new_order = list(map(itemgetter(1), zero_ocurrences))
    old_order = np.argsort(new_order)

    return new_order, old_order

def gauss_solver(matrix,end_matrix):
    # zmieniamy kopię macierzy

    matrix_1 = matrix.copy()
    end_matrix_1 = end_matrix.copy()

    for i in range(matrix_1.shape[0]):

        sub_matrix = matrix_1[i:, :]
        new_order,_ = rows_permutations(sub_matrix)
        full_order = [i + idx for idx in new_order]

        matrix_1[i:] = matrix_1[full_order]
        end_matrix_1[i:] = end_matrix_1[full_order]

        other_rows=list(range(i))
        other_rows.extend(list(range(i+1,matrix_1.shape[0])))

        div=1/matrix_1[i,i]
        matrix_1[i]=matrix_1[i]*div
        end_matrix_1[i]=end_matrix_1[i]*(div)
        
        for j in other_rows:
            div=matrix_1[j,i]
            matrix_1[j]=matrix_1[j]-(div*matrix_1[i])
            end_matrix_1[j]=end_matrix_1[j]-(div*end_matrix_1[i])
    return end_matrix_1




In [170]:
n = 5
A = np.array([[1.0 / (i + j - 1) for j in range(1, n + 1)] for i in range(1, n + 1)], dtype=np.float32)
b = np.array([5.0, 4.0, 3.0, 2.0, 1.0], dtype=np.float32).transpose()

x = gauss_solver(A, b)
print("Rozwiązanie początkowe x^(1):\n", x)


for k in range(1_000_000):
    r = b.astype(np.float64) - np.linalg.matmul(A, x,).astype(np.float64)
    
    res_norm = np.max(np.abs(r))
    print(f"norma reszty = {res_norm}")
    
    if res_norm < 1e-7:  
        break
        
    dx = gauss_solver(A, r)
    
    x = x + dx

print("Ostateczne poprawione rozwiązanie:\n", x)

Rozwiązanie początkowe x^(1):
 [   -94.941315   2158.9722   -10705.703     17913.64      -9446.933   ]
norma reszty = 0.0006556510925292969
norma reszty = 9.667212366526456e-05
norma reszty = 1.3520273878242506e-07
norma reszty = 1.000035609877159e-10
Ostateczne poprawione rozwiązanie:
 [   -94.84781942   2157.10127308 -10697.32231983  17900.65789887
  -9440.4652541 ]


### Zadanie 2
2. Napisz program rozwiązujący poniższy układ równań metodą Gaussa-
Seidla:
$$\begin{pmatrix}
4 & -1 & 0 & \cdots & 0 & 0 & 1 \\
-1 & 4 & -1 & \cdots & 0 & 0 & 0 \\
0 & -1 & 4 & \cdots & 0 & 0 & 0\\
\vdots & \vdots & \vdots & \ddots & \vdots & \vdots & \vdots \\
0 & 0 & 0 & \cdots & 4 & -1 & 0\\
0 & 0 & 0 & \cdots & -1 & 4 & -1\\
1 & 0 & 0 & \cdots & 0 & -1 & 4
\end{pmatrix}
$$
Program powinien działać dla dowolnych n. Przeprowadź obliczenia dla
n = 20 i sprawdź zbieżność rozwiązania.

In [171]:
def gen_func(k,n):
    out=[0 for i in range(n)]
    out[k] = 4
    try:
        out[k-1]=-1
    except:
        pass
    try:
        out[k+1]=-1
    except:
        pass
    if k==0:
        out[n-1]=1
    if k==n-1:
        out[0]=1           
    return out
n=20
A=np.matrix([gen_func(i,n) for i in range(n)])
b=[0 for i in range(n-1)]
b.append(100)
b=np.matrix(b).T
def check_rest(new_list,old_list,threshold_amount):
    new_list=np.asmatrix(new_list.copy())
    old_list=np.asmatrix(old_list.copy())
    old_list=old_list*-1
    res=np.add(new_list,old_list)
    def threshold(t,val):
        if abs(val)<t:
            return True
        if abs(val)>t:
            return False
    o=[threshold(res[i,0],threshold_amount) for i in range(len(new_list))]
    return np.all(o)



def gauss_seidel_solver(start_matrix,goal_matrix):
    x_now=[0 for i in range(start_matrix.shape[0])]
    x_past=[]
    for i in range(1_000_000):
        x_past=x_now.copy()
        for j in range(n):
            w_n = (np.dot(np.asmatrix(x_now.copy()),start_matrix[j].T))[0,0]
            x_now[j]=(1/start_matrix[j,j])*(goal_matrix[j,0] - w_n  + (start_matrix[j,j]*x_now[j]))
            #print(j,x_now)    
        if check_rest(x_now,x_past,1e-14)==True:
            break
    return x_now    
result=np.asmatrix(gauss_seidel_solver(A,b))
print(result.T,end="\n\n")
print(np.matmul(A,result.T))  

[[-7.73502692e+00]
 [-2.07259422e+00]
 [-5.55349942e-01]
 [-1.48805551e-01]
 [-3.98722579e-02]
 [-1.06834772e-02]
 [-2.86164694e-03]
 [-7.63106890e-04]
 [-1.90777544e-04]
 [-8.89354711e-10]
 [ 1.90775724e-04]
 [ 7.63104969e-04]
 [ 2.86164492e-03]
 [ 1.06834752e-02]
 [ 3.98722561e-02]
 [ 1.48805549e-01]
 [ 5.55349941e-01]
 [ 2.07259421e+00]
 [ 7.73502692e+00]
 [ 2.88675135e+01]]

[[-4.54676297e-10]
 [-1.18855503e-09]
 [-2.18612728e-09]
 [-3.14488313e-09]
 [-3.77465639e-09]
 [-3.93221338e-09]
 [-3.65144036e-09]
 [-3.08093832e-09]
 [-2.39629808e-09]
 [-1.73732066e-09]
 [-1.18453844e-09]
 [-7.65014987e-10]
 [-4.70778659e-10]
 [-2.77423216e-10]
 [-1.57206470e-10]
 [-8.59723404e-11]
 [-4.55151472e-11]
 [-2.33892905e-11]
 [ 1.10418341e-11]
 [ 1.00000000e+02]]


Rozwiązanie jest zbierzne bo jest symetryczna i dodatnio określona oraz silnie diagonalnie dominująca